In [2]:
import os
import sys
import itertools
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold, train_test_split
from sklearn.preprocessing import StandardScaler

# --- Path Configuration ---
root_dir = os.path.abspath("..")
paths_to_add = [
    root_dir,
    os.path.abspath("../current_setpoints"),
    os.path.abspath("../current_setpoints/optimization"),
    os.path.abspath("../current_setpoints/model")
]

for path in paths_to_add:
    if path not in sys.path:
        sys.path.append(path)

# --- Custom Library Imports ---
from current_setpoints.data import FluxValues, IEEEMachine2
from current_setpoints.utils import NeuralTorquePredictor, load_aggregated_csv_data
from trainer_utils import train_model, prepare_fold_dataloaders, evaluate_model

ModuleNotFoundError: No module named 'plot_config'

In [ ]:
# --- File Paths & Mappings ---
AGGREGATED_FILE_PATH = '../data/aggregated_file_means.csv' 
TARGET_VARIABLE = 'torq'
COLUMN_MAP = {'omega': 'omega', 'id1': 'id1', 'iq1': 'iq1', 'id3': 'id3', 'iq3': 'iq3', 'torq': 'torq'}
INPUT_SIZE = 5

# --- Phase 1: CV Hyperparameters ---
K_SPLITS = 5
TEST_SIZE = 0.15
CV_EPOCHS = 200
CV_PATIENCE = 10

HIDDEN_SIZES = [6, 8, 12, 16]
LEARNING_RATES = [3e-3, 5e-3, 8e-3, 1e-2, 1.5e-2]
REG_LAMBDAS = [5e-5, 1e-4, 2e-4]

# --- Phase 2: Final Training Settings ---
FINAL_EPOCHS = 800
FINAL_BATCH_SIZE = 64
FINAL_PATIENCE = 15
MIN_DELTA = 1e-5
MODEL_SAVE_PATH = 'NTM_Best_Model.pth'
SCALER_SAVE_PATH = 'NTM_Best_Scaler.npy'

# --- Device Setup ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {DEVICE}") 

# --- Analytical Model Setup ---
flux_values = FluxValues()
machine = IEEEMachine2(flux_values)

# --- Data Loading ---
data = load_aggregated_csv_data(AGGREGATED_FILE_PATH, COLUMN_MAP)
X_features = ['omega', 'id1', 'iq1', 'id3', 'iq3']

X = data[X_features].values.astype(np.float32)
y = data[[TARGET_VARIABLE]].values.astype(np.float32)

# --- OPTION B: Pre-calculate Dynamic B vectors ---
print("Pre-calculating dynamic B vectors based on machine states...")
B_list = []
for row in X:
    omega_val = row[0]
    currents_val = row[1:] 
    machine.update_state(omega=omega_val, vec_curr_dq=currents_val)
    B_list.append(machine.vec_b.copy())

B_array = np.array(B_list).astype(np.float32)

# Global Train/Test Split (Creates the Holdout Test Set) - NOW INCLUDES B_array
X_train_val, X_test, y_train_val, y_test, B_train_val, B_test = train_test_split(
    X, y, B_array, test_size=TEST_SIZE, random_state=42
)

print(f"Training/Validation Base Set Size: {X_train_val.shape[0]}")
print(f"Holdout Test Set Size: {X_test.shape[0]}")

Using compute device: cpu
Loaded 174 valid data points from CSV.
Pre-calculating dynamic B vectors based on machine states...
Training/Validation Base Set Size: 147
Holdout Test Set Size: 27


In [ ]:

# --- Cross-Validation Execution ---
kf = KFold(n_splits=K_SPLITS, shuffle=True, random_state=42)
hyperparameters = list(itertools.product(HIDDEN_SIZES, LEARNING_RATES, REG_LAMBDAS))
results_list = []

print(f"Total Combinations to Test: {len(hyperparameters)}")

for iteration, (h_size, lr, reg) in enumerate(hyperparameters):
    print(f"\n*** Combination {iteration + 1}/{len(hyperparameters)} | H_Size: {h_size}, LR: {lr:.1e}, Reg: {reg:.1e} ***")
    cv_performance = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_val)):
        # Pass the B arrays into the dataloader preparation
        train_loader, val_loader, scaler_X = prepare_fold_dataloaders(
            X_train_val[train_idx], X_train_val[val_idx], 
            y_train_val[train_idx], y_train_val[val_idx],
            B_train_val[train_idx], B_train_val[val_idx]
        )

        model = NeuralTorquePredictor(
            input_size=INPUT_SIZE, hidden_size=h_size, 
            scaler_X=scaler_X, machine=machine, device=DEVICE
        ).to(DEVICE)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=reg)
        criterion = nn.MSELoss()

        # machine argument removed; handled dynamically by dataloaders
        best_loss, _ = train_model(
            model, train_loader, val_loader, criterion, optimizer, 
            CV_EPOCHS, CV_PATIENCE, DEVICE
        )
        cv_performance.append(np.sqrt(best_loss))
        
    mean_rmse, std_rmse = np.mean(cv_performance), np.std(cv_performance)
    results_list.append({'H_Size': h_size, 'LR': lr, 'Reg_Lambda': reg, 'Mean_CV_RMSE': mean_rmse})
    print(f"  --> Mean CV RMSE: {mean_rmse:.4f} (+/- {std_rmse:.4f})")

# --- Extract Optimal Parameters ---
results_df = pd.DataFrame(results_list).sort_values(by='Mean_CV_RMSE')
best_params = results_df.iloc[0]

OPTIMAL_HIDDEN_SIZE = int(best_params['H_Size'])
OPTIMAL_LEARNING_RATE = best_params['LR']
OPTIMAL_REG_LAMBDA = best_params['Reg_Lambda']

print("\n=======================================================================")
print(f"BEST MEAN CV RMSE Found: {best_params['Mean_CV_RMSE']:.4f} Nm")
print(f"Optimal Params -> H_Size: {OPTIMAL_HIDDEN_SIZE}, LR: {OPTIMAL_LEARNING_RATE:.1e}, Reg: {OPTIMAL_REG_LAMBDA:.1e}")
print("=======================================================================")

display(results_df.head()) # 'display' looks cleaner in Jupyter than print 


Total Combinations to Test: 60

*** Combination 1/60 | H_Size: 6, LR: 3.0e-03, Reg: 5.0e-05 ***

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 111!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 70!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 130!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 39!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 34!
  --> Mean CV RMSE: 0.0592 (+/- 0.0202)

*** Combination 2/60 | H_Size: 6, LR: 3.0e-03, Reg: 1.0e-04 ***

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 105!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 95!

Starting Training with Early Stopping (Patience=10)...

Early stopping triggered at epoch 61!

Starting Training with Early Stoppi

,H_Size,LR,Reg_Lambda,Mean_CV_RMSE
57,16,0.015,0.00005,0.037062
44,12,0.015,0.00020,0.037288
59,16,0.015,0.00020,0.037574
52,16,0.008,0.00010,0.038012
43,12,0.015,0.00010,0.038569


In [ ]:
# --- Final Data Preparation ---
# Split X_train_val again to get a dedicated early-stopping validation set for final training
X_train, X_val, y_train, y_val, B_train, B_val = train_test_split(
    X_train_val, y_train_val, B_train_val, test_size=0.20, random_state=42
)

# Fit scaler ONLY on the final training set
scaler_X = StandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm = scaler_X.transform(X_val)
X_test_norm = scaler_X.transform(X_test)

# Create PyTorch DataLoaders (Now including B_tensors)
train_dataset = TensorDataset(
    torch.from_numpy(X_train_norm).float(),
    torch.from_numpy(y_train).float(),
    torch.from_numpy(B_train).float(),
)
val_dataset = TensorDataset(
    torch.from_numpy(X_val_norm).float(),
    torch.from_numpy(y_val).float(),
    torch.from_numpy(B_val).float(),
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test_norm).float(),
    torch.from_numpy(y_test).float(),
    torch.from_numpy(B_test).float(),
)

train_loader = DataLoader(train_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=FINAL_BATCH_SIZE, shuffle=False)

# --- Initialize Final Model (Using Optimals from Phase 1) ---
final_model = NeuralTorquePredictor(
    input_size=INPUT_SIZE,
    hidden_size=OPTIMAL_HIDDEN_SIZE,
    scaler_X=scaler_X,
    machine=machine,
    device=DEVICE,
).to(DEVICE)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    final_model.parameters(), lr=OPTIMAL_LEARNING_RATE, weight_decay=OPTIMAL_REG_LAMBDA
)

# --- Final Training Execution ---
print(
    f"\nStarting Final Training with H_Size: {OPTIMAL_HIDDEN_SIZE}, LR: {OPTIMAL_LEARNING_RATE:.1e}, Reg: {OPTIMAL_REG_LAMBDA:.1e}"
)

# machine argument removed; handled dynamically by dataloaders
best_val_loss, epochs_run = train_model(
    model=final_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=FINAL_EPOCHS,
    patience=FINAL_PATIENCE,
    device=DEVICE,
    min_delta=MIN_DELTA,
    verbose=True,
)

# --- Save Artifacts ---
torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
scaler_data = {"mean": scaler_X.mean_, "scale": scaler_X.scale_}
np.save(SCALER_SAVE_PATH, np.array(scaler_data, dtype=object), allow_pickle=True)

# --- Final Evaluation ---
# machine argument removed; handled dynamically by dataloaders
final_test_rmse = evaluate_model(final_model, test_loader, criterion, DEVICE)

print("\n=======================================================================")
print("            ✅ FINAL NTM MODEL PERFORMANCE REPORT ✅")
print("=======================================================================")
print(
    f"Training completed in {epochs_run} epochs. Best Validation Loss: {best_val_loss:.6f}"
)
print(f"FINAL TEST SET RMSE (Generalization Metric): {final_test_rmse:.4f} Nm")
print("=======================================================================")


Starting Final Training with H_Size: 16, LR: 1.5e-02, Reg: 5.0e-05

Starting Training with Early Stopping (Patience=15)...

Early stopping triggered at epoch 35!

            ✅ FINAL PIRN MODEL PERFORMANCE REPORT ✅
Training completed in 35 epochs. Best Validation Loss: 0.000972
FINAL TEST SET RMSE (Generalization Metric): 0.0374 Nm
